In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Dataset Deep Exploration

In [ ]:
import os
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# === SET YOUR DATASET ROOT HERE ===
DATASET_ROOT = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TEXT_EXTS = {".txt", ".md", ".pdf"}

print(f"Dataset root: {DATASET_ROOT}")
print(f"Exists: {os.path.isdir(DATASET_ROOT)}")

: 

In [ ]:
# Recursive folder exploration
all_image_paths = []
all_text_paths = []
folder_records = []
ext_counter = Counter()

for dirpath, dirnames, filenames in os.walk(DATASET_ROOT):
    img_count = 0
    txt_count = 0
    other_count = 0

    for fname in filenames:
        ext = Path(fname).suffix.lower()
        ext_counter[ext] += 1
        full_path = os.path.join(dirpath, fname)

        if ext in IMAGE_EXTS:
            img_count += 1
            all_image_paths.append(full_path)
        elif ext in TEXT_EXTS:
            txt_count += 1
            all_text_paths.append(full_path)
        else:
            other_count += 1

    if filenames:
        rel = os.path.relpath(dirpath, DATASET_ROOT)
        folder_records.append({
            "folder": rel,
            "images": img_count,
            "text_files": txt_count,
            "other": other_count,
            "total": img_count + txt_count + other_count,
        })

df = pd.DataFrame(folder_records)
display(df)

print(f"\n{'='*50}")
print(f"Grand total files : {len(all_image_paths) + len(all_text_paths) + df['other'].sum()}")
print(f"  Images           : {len(all_image_paths)}")
print(f"  Text files       : {len(all_text_paths)}")
print(f"  Other            : {df['other'].sum()}")
print(f"Folders with files : {len(df)}")
print(f"\nFile extension breakdown:")
for ext, count in ext_counter.most_common():
    print(f"  {ext or '(no ext)':>10} : {count}")

: 

In [ ]:
# Content verification

# --- Image sample ---
if all_image_paths:
    sample_img_path = all_image_paths[0]
    img = Image.open(sample_img_path)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    ax.set_title(os.path.basename(sample_img_path), fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    file_size = os.path.getsize(sample_img_path)
    print(f"Path       : {sample_img_path}")
    print(f"Dimensions : {img.width} x {img.height}")
    print(f"Mode       : {img.mode}")
    print(f"File size  : {file_size / 1024:.1f} KB")
else:
    print("No images found.")

print()

# --- Text sample ---
if all_text_paths:
    sample_txt_path = all_text_paths[0]
    print(f"Text file  : {os.path.basename(sample_txt_path)}")
    print(f"Path       : {sample_txt_path}")
    with open(sample_txt_path, "r", encoding="utf-8", errors="replace") as f:
        preview = f.read(200)
    print(f"Preview    :\n{preview}")
else:
    print("No text files found.")

# NewsClipPings Dataset: How Images Link to Articles

The dataset uses a **two-file system**:
1. **`merged_balanced/train.json`** — contains annotation pairs (the task labels)
2. **`metadata/test.json`** (and similar) — contains the actual article info (caption, image path, article path)

The `id` field in annotations maps to a key in the metadata file to retrieve the caption, image, and article.

In [ ]:
import json

# ── Paths ──
MERGED_TRAIN = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'merged_balanced', 'train.json')
METADATA     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'metadata', 'test.json')

# ── 1. Load merged_balanced/train.json (annotations) ──
with open(MERGED_TRAIN, "r", encoding="utf-8") as f:
    train_data = json.load(f)

source_datasets = train_data["source_datasets"]
annotations     = train_data["annotations"]

print(f"Source datasets mapping: {source_datasets}")
print(f"Total annotations: {len(annotations):,}")
print()

# Show first 6 entries (3 real + 3 falsified pairs)
print("=" * 70)
print("SAMPLE ANNOTATIONS (first 6 entries):")
print("=" * 70)
for ann in annotations[:6]:
    label = "REAL (original pair)" if not ann["falsified"] else "FAKE (swapped image)"
    src   = source_datasets[str(ann["source_dataset"])]
    print(f"\n  id            : {ann['id']}")
    print(f"  image_id      : {ann['image_id']}")
    print(f"  falsified     : {ann['falsified']}  ← {label}")
    print(f"  similarity    : {ann['similarity_score']:.4f}")
    print(f"  source_dataset: {ann['source_dataset']} → {src}")
    print(f"  ---")

print("\n" + "=" * 70)
print("FIELD EXPLANATIONS:")
print("=" * 70)
print("""
• id             → The article/caption ID. Use this to look up the 
                   caption and article text in the metadata file.

• image_id       → The image ID. When falsified=False, image_id == id 
                   (the original image). When falsified=True, image_id 
                   is a DIFFERENT article's image swapped in to create 
                   a mismatched pair.

• falsified      → False = real pair (image belongs to this article)
                   True  = fake pair (image stolen from another article)

• similarity_score → How similar the swapped image is to the original.
                     1.0 = identical (real pairs). Lower = less similar.

• source_dataset → Which method was used to find the similar swap:
                   0 = CLIP text-image similarity
                   1 = CLIP text-text similarity  
                   2 = ResNet place/scene similarity
                   3 = SBERT text-text (person-based)
""")

: 

In [ ]:
# ── 2. Load metadata to see what each ID resolves to ──
with open(METADATA, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Metadata entries: {len(metadata):,}")
print()

# Show a sample metadata entry
sample_key = list(metadata.keys())[0]
sample = metadata[sample_key]

print("=" * 70)
print(f"SAMPLE METADATA ENTRY (id={sample_key}):")
print("=" * 70)
for k, v in sample.items():
    if isinstance(v, list) and len(v) > 2:
        print(f"  {k:25s}: {v[:2]} ... ({len(v)} items)")
    else:
        print(f"  {k:25s}: {v}")

print()
print("=" * 70)
print("METADATA FIELD EXPLANATIONS:")
print("=" * 70)
print("""
• id                → Unique article ID (matches 'id' in annotations)
• caption           → The image caption / headline text
• image_path        → Relative path to the image file
                      (prefix: visual_news/ → maps to dataset/origin/)
• article_path      → Relative path to the article text file
• topic             → News topic (e.g., football, politics)
• source            → News outlet (bbc, guardian, usa_today, washington_post)
• timestamp         → Publication date
• image_has_person  → Whether a person is detected in the image
• caption_entities_* → Named entities extracted from the caption
""")

# ── 3. Show how an annotation connects to metadata ──
print("=" * 70)
print("HOW AN ANNOTATION CONNECTS TO AN ARTICLE + IMAGE:")
print("=" * 70)

# Pick the first annotation pair (real + fake)
real_ann = annotations[0]  # falsified=False
fake_ann = annotations[1]  # falsified=True

# The article caption comes from the 'id' field
article_id = str(real_ann["id"])
if article_id in metadata:
    meta = metadata[article_id]
    print(f"\n  Article ID {article_id}:")
    print(f"    Caption    : {meta['caption'][:100]}...")
    print(f"    Image path : {meta['image_path']}")
    print(f"    Source     : {meta['source']}")
else:
    print(f"\n  (ID {article_id} not in metadata/test.json — it's in a different split's metadata)")
    # Demo with a key that IS in metadata
    demo_key = list(metadata.keys())[0]
    meta = metadata[demo_key]
    print(f"\n  Demo with ID {demo_key} from metadata:")
    print(f"    Caption    : {meta['caption'][:100]}")
    print(f"    Image path : {meta['image_path']}")
    print(f"    Source     : {meta['source']}")

print(f"""
  ┌─────────────────────────────────────────────────┐
  │  LINKING LOGIC:                                 │
  │                                                 │
  │  annotation['id']       → metadata[id]['caption']│
  │                         → metadata[id]['image_path'] (original image)│
  │                                                 │
  │  annotation['image_id'] → metadata[image_id]['image_path']│
  │                           (the actual image used)│
  │                                                 │
  │  If falsified=False: id == image_id (match)     │
  │  If falsified=True:  id != image_id (mismatch!) │
  └─────────────────────────────────────────────────┘
""")